<a href="https://colab.research.google.com/github/Sayrinix/Entrega_Final_CDD_Luciano_Sayrina/blob/main/notebooks/S1_Luciano_Sayrina_CDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio 1 — Obtención del dato
**Ciencia de Datos · Ingeniería de Ejecución en Informática — Advance UCSC**

| | |
|---|---|
| **Semana** | 1 |
| **Resultado de aprendizaje** | RA1 (foco) · RA3 (transversal) · RA5 (apertura) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Estudiante** | _Sayrina Luciano Jofré_ |

---

## Antes de empezar

Sube al entorno los tres archivos que acompañan esta guía:

- `S1_Dataset_Guia_CDD.csv`
- `S1_Base_Ejemplo_CDD.sqlite`
- `S1_Respuesta_API_Ejemplo_CDD.json`

En Google Colab: panel izquierdo → icono de carpeta → *Subir*. En Jupyter local: déjalos en la misma carpeta que este cuaderno.

## Celda 0 · Pregunta de activación

Antes de escribir código, responde aquí para ti (no se entrega):

> **¿Qué dato de tu propio trabajo no usarías hoy para tomar una decisión importante, y por qué?**

_Actualmente me encuentro desempleada, pero considero que no utilizaría datos personales sensibles o que permitan identificar directamente a una persona, como su RUT, nombre completo, dirección o información médica, sin una finalidad legítima y las medidas de protección correspondientes.

Tampoco utilizaría datos incompletos, desactualizados o que puedan introducir sesgos en un análisis, ya que podrían generar interpretaciones erróneas y afectar la toma de decisiones. Por ejemplo, en el área de eCommerce, utilizar registros de ventas duplicados o incorrectos podría alterar los indicadores de rendimiento de una empresa_

## 1 · Preparación del entorno

Un flujo reproducible parte por declarar qué versiones se usaron. Si alguien ejecuta este cuaderno dentro de un año y obtiene otro resultado, lo primero que va a mirar es esto.

In [4]:
import sys, platform
import pandas as pd
import numpy as np

RUTA = ''          # en Colab déjalo vacío; en local, la carpeta donde están los archivos
SEMILLA = 20261
np.random.seed(SEMILLA)

print('Python    :', sys.version.split()[0], '·', platform.system())
print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('semilla   :', SEMILLA)

Python    : 3.13.18 · Linux
pandas    : 2.2.3
numpy     : 2.4.6
semilla   : 20261


## 2 · Fuente 1 — Archivo plano (CSV)

El primer problema de un archivo plano nunca es el análisis: es **abrirlo bien**. Tres cosas pueden salir mal antes de ver un solo número: la codificación, el separador y los tipos de dato.

Ejecuta la celda siguiente y observa el resultado. Después responde: ¿cuál es el separador de este archivo?

In [5]:
# Lectura deliberadamente ingenua: mira lo que pasa
try:
    crudo = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv')
    print('Forma:', crudo.shape)
    display(crudo.head(3))
except Exception as e:
    print(type(e).__name__)
    print(e)

ParserError
Error tokenizing data. C error: Expected 1 fields in line 39, saw 2



**Lee el error, no lo saltes.** Dice algo como *«Expected 1 fields in line 39, saw 2»*.

Se traduce así: pandas asumió que el separador era la coma, no encontró ninguna en la primera línea y concluyó que el archivo tenía **una sola columna**. Todo iba bien hasta la línea 39, donde apareció una coma —un decimal escrito a la chilena, `12,4`— y esa fila pasó a tener dos campos. Contradicción, y se detiene.

Son dos defectos a la vez: el separador real es `;` y además hay decimales con coma. Resolvamos primero el separador.

In [6]:
aire = pd.read_csv(
    RUTA + 'S1_Dataset_Guia_CDD.csv',
    sep=';',
    encoding='utf-8',
)
print('Forma:', aire.shape)
aire.head()

Forma: (734, 11)


,id_registro,estacion_id,estacion,comuna,region,fecha,pm25,pm10,temperatura_c,humedad_pct,validado
0,02-20260626,2,Los Ángeles Oriente,Los Ángeles,Biobío,2026-06-26,77.1,154.9,9.5,58.3,SI
1,08-20260405,8,Coronel Norte,Coronel,Biobío,2026-04-05,31.6,52.0,12.7,65.4,SI
2,06-20260621,6,Osorno Entre Lagos,Osorno,Los Lagos,2026-06-21,63.5,111.4,6.7,83.0,SI
3,01-20260531,1,Kingston,Concepción,Biobío,2026-05-31,48.1,79.4,14.4,75.1,SI
4,05-20260522,5,Temuco Padre Las Casas,Temuco,Araucanía,2026-05-22,85.0,162.1,11.5,82.9,SI


In [7]:
# Tipos de dato con los que pandas interpretó cada columna
aire.dtypes

,0
id_registro,object
estacion_id,int64
estacion,object
comuna,object
region,object
fecha,object
pm25,float64
pm10,float64
temperatura_c,object
humedad_pct,float64


### ✏️ Actividad 2.1

Mira los tipos con atención. `pm25` y `pm10` quedaron numéricos, pero `temperatura_c` quedó como `object`, es decir, texto. Que una columna numérica quede como texto **siempre** significa que hay algo dentro que no es un número: aquí son los nueve decimales escritos con coma que provocaron el error anterior.

Y existe la trampa inversa: `pm10` es numérico y aun así está contaminado. Lo verás en la actividad 3.2.

**Escribe el código** que muestre los valores no numéricos de `temperatura_c`. Pista: `pd.to_numeric(..., errors='coerce')` convierte a `NaN` lo que no puede convertir.

In [8]:
# Tu código aquí
conversion = pd.to_numeric(aire["temperatura_c"], errors="coerce")
mascara = conversion.isna()
print(aire.loc[mascara, "temperatura_c"])

37      7,8
209     8,3
347    15,0
429    10,5
522     8,8
563    13,3
603    16,4
618    10,1
702    10,4
Name: temperatura_c, dtype: object


## 3 · Perfilado inicial: ¿qué tengo entre manos?

Antes de analizar, hay que **conocer el dato**. Estas cuatro celdas son el mínimo que un analista ejecuta siempre, en este orden.

In [9]:
# a) Dimensiones y nombres de columna
print('Filas   :', aire.shape[0])
print('Columnas:', aire.shape[1])
print()
print(aire.columns.tolist())

Filas   : 734
Columnas: 11

['id_registro', 'estacion_id', 'estacion', 'comuna', 'region', 'fecha', 'pm25', 'pm10', 'temperatura_c', 'humedad_pct', 'validado']


In [10]:
# b) Valores faltantes por columna
faltantes = aire.isna().sum().to_frame('faltantes')
faltantes['porcentaje'] = (faltantes['faltantes'] / len(aire) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,porcentaje
pm25,35,4.77
estacion_id,0,0.00
id_registro,0,0.00
estacion,0,0.00
comuna,0,0.00
region,0,0.00
fecha,0,0.00
pm10,0,0.00
temperatura_c,0,0.00
humedad_pct,0,0.00


In [11]:
# c) Filas duplicadas completas
print('Duplicados exactos:', aire.duplicated().sum())
print('id_registro repetidos:', aire['id_registro'].duplicated().sum())

Duplicados exactos: 14
id_registro repetidos: 14


In [12]:
# d) Cardinalidad de las columnas categóricas
for col in ['estacion', 'comuna', 'region', 'validado']:
    print(f'{col:15s} · {aire[col].nunique():3d} valores distintos')

estacion        ·   8 valores distintos
comuna          ·  16 valores distintos
region          ·   5 valores distintos
validado        ·   1 valores distintos


### ✏️ Actividad 3.1

La columna `comuna` declara más valores distintos de los que deberían existir. **Ejecuta** `aire['comuna'].unique()` y explica en la celda de texto siguiente por qué ocurre.

Ojo: no lo corrijas todavía. Esta semana **detectamos**; en la semana 2 **corregimos** y dejamos registrada la decisión en la bitácora.

In [13]:
aire['comuna'].unique()

array(['Los Ángeles', 'Coronel', 'Osorno', 'Concepción', 'Temuco',
       'Talcahuano', 'Chillán', 'Valdivia', 'TEMUCO  ', 'VALDIVIA  ',
       'LOS ÁNGELES  ', 'TALCAHUANO  ', 'CHILLÁN  ', 'CONCEPCIÓN  ',
       'CORONEL  ', 'OSORNO  '], dtype=object)

_Tu explicación aquí:_ Existen diferentes formas de escribir una misma comuna, en un comportamiento conocido como case sensitivity, donde algunas comunas están escritas en mayúsculas y .unique las identifica como valores diferentes. Se puede producir una inconsistencia en los datos, donde se altere la cantidad de categorías.



### ✏️ Actividad 3.2 — Valores centinela

Un **valor centinela** es un número real que en realidad significa «no hay dato»: `-999`, `0`, `9999`. Si no lo detectas, entra al promedio y lo arruina.

**Escribe el código** que cuente cuántas veces aparece `-999` en `pm10` y calcula el promedio de `pm10` con y sin esos valores. Anota la diferencia.

In [14]:
# Tu código aquí
cantidad = (aire["pm10"] == -999).sum()
print("Cantidad de centinelas:", cantidad)

promedio_original = aire["pm10"].mean()
print("Promedio original:", promedio_original)

promedio_sin_centinela = aire.loc[aire["pm10"] != -999, "pm10"].mean()
print("Promedio sin centinelas:", promedio_sin_centinela)

Cantidad de centinelas: 18
Promedio original: 77.09904632152588
Promedio sin centinelas: 104.15181564245812


## 4 · Fuente 2 — Base de datos relacional (SQL)

Buena parte de los datos de una organización no está en archivos, sino en una base de datos. El archivo `S1_Base_Ejemplo_CDD.sqlite` contiene tres tablas: `estacion`, `medicion` y `episodio`.

SQLite viene incluido en Python: no hay que instalar nada.

In [15]:
import sqlite3

con = sqlite3.connect(RUTA + 'S1_Base_Ejemplo_CDD.sqlite')

# ¿Qué tablas hay?
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", con)

,name
0,estacion
1,medicion
2,sqlite_sequence
3,episodio


In [16]:
# Estructura de una tabla
pd.read_sql_query('PRAGMA table_info(medicion)', con)

,cid,name,type,notnull,dflt_value,pk
0,0,medicion_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,pm25,REAL,0,None,0
4,4,pm10,REAL,0,None,0
5,5,temperatura_c,REAL,0,None,0
6,6,humedad_pct,REAL,0,None,0


In [17]:
# Una consulta con JOIN y agregación: promedio de PM2.5 por comuna
consulta = '''
SELECT  e.comuna,
        e.region,
        COUNT(*)              AS n_mediciones,
        ROUND(AVG(m.pm25), 1) AS pm25_promedio,
        ROUND(MAX(m.pm25), 1) AS pm25_maximo
FROM    medicion m
JOIN    estacion e ON e.estacion_id = m.estacion_id
GROUP BY e.comuna, e.region
ORDER BY pm25_promedio DESC
'''
pd.read_sql_query(consulta, con)

,comuna,region,n_mediciones,pm25_promedio,pm25_maximo
0,Temuco,Araucanía,90,74.2,102.1
1,Chillán,Ñuble,90,65.8,98.5
2,Los Ángeles,Biobío,90,62.2,87.5
3,Osorno,Los Lagos,90,60.3,82.5
4,Valdivia,Los Ríos,90,53.9,78.0
5,Coronel,Biobío,90,49.0,76.5
6,Concepción,Biobío,90,45.8,70.7
7,Talcahuano,Biobío,90,41.3,67.8


### ✏️ Actividad 4.1

**Escribe una consulta SQL** que responda: *¿cuántos episodios de cada categoría (`Alerta`, `Preemergencia`, `Emergencia`) registró cada comuna?*

Necesitas unir `episodio` con `estacion` y agrupar por dos columnas.

In [18]:
pd.read_sql_query('PRAGMA table_info(episodio)', con)

,cid,name,type,notnull,dflt_value,pk
0,0,episodio_id,INTEGER,0,None,1
1,1,estacion_id,INTEGER,1,None,0
2,2,fecha,TEXT,1,None,0
3,3,categoria,TEXT,1,None,0


In [19]:

consulta_2 = '''
SELECT
    e.comuna,
    ep.categoria,
    COUNT(*) AS cantidad_episodios
FROM episodio ep
JOIN estacion e
    ON e.estacion_id = ep.estacion_id
GROUP BY
    e.comuna,
    ep.categoria
ORDER BY
    cantidad_episodios DESC;
'''
pd.read_sql_query(consulta_2, con)

,comuna,categoria,cantidad_episodios
0,Temuco,Alerta,25
1,Los Ángeles,Alerta,6
2,Chillán,Alerta,5
3,Osorno,Alerta,5
4,Temuco,Preemergencia,4
5,Chillán,Preemergencia,1


In [20]:
con.close()

## 5 · Fuente 3 — Servicio web (API) y JSON anidado

Una API devuelve **JSON**, no una tabla. El JSON es jerárquico: dentro de un registro puede haber otro objeto, y dentro de ese, una lista. Convertirlo en una tabla rectangular es parte del trabajo.

Usamos aquí la respuesta guardada en archivo. En un caso real la obtendrías con `requests.get(url).json()`; el archivo existe para que el laboratorio funcione aunque el servicio esté caído.

In [21]:
import json

with open(RUTA + 'S1_Respuesta_API_Ejemplo_CDD.json', encoding='utf-8') as fh:
    respuesta = json.load(fh)

print('Claves de primer nivel:', list(respuesta.keys()))
print()
print('Metadatos:')
for k, v in respuesta['meta'].items():
    print(f'  {k}: {v}')

Claves de primer nivel: ['meta', 'data']

Metadatos:
  servicio: API de ejemplo · Red de monitoreo de calidad del aire (simulada para docencia)
  version: 1.2
  generado: 2026-09-21T10:00:00-03:00
  licencia: CC BY 4.0 (conjunto sintético de uso docente)
  paginacion: {'pagina': 1, 'por_pagina': 100, 'total': 100, 'siguiente': None}


In [22]:
# Un registro cualquiera, para entender la estructura
print(json.dumps(respuesta['data'][0], ensure_ascii=False, indent=2))

{
  "codigo_estacion": "EST-001",
  "ubicacion": {
    "comuna": "Concepción",
    "region": "Biobío",
    "coordenadas": {
      "lat": -36.827,
      "lon": -73.0498
    }
  },
  "fecha": "2026-04-01",
  "mediciones": [
    {
      "parametro": "pm25",
      "valor": 37.6,
      "unidad": "ug/m3"
    },
    {
      "parametro": "pm10",
      "valor": 58.8,
      "unidad": "ug/m3"
    },
    {
      "parametro": "temperatura",
      "valor": 13.1,
      "unidad": "C"
    }
  ],
  "calidad_dato": "preliminar"
}


In [23]:
# json_normalize aplana la jerarquía: fíjate en los nombres de columna que genera
api = pd.json_normalize(respuesta['data'])
api.head(3)

,codigo_estacion,fecha,mediciones,calidad_dato,ubicacion.comuna,ubicacion.region,ubicacion.coordenadas.lat,ubicacion.coordenadas.lon
0,EST-001,2026-04-01,"[{'parametro': 'pm25', 'valor': 37.6, 'unidad'...",preliminar,Concepción,Biobío,-36.827,-73.0498
1,EST-001,2026-04-02,"[{'parametro': 'pm25', 'valor': 25.6, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498
2,EST-001,2026-04-03,"[{'parametro': 'pm25', 'valor': 29.3, 'unidad'...",validado,Concepción,Biobío,-36.827,-73.0498


In [24]:
# La lista 'mediciones' sigue anidada. record_path la despliega en filas:
api_largo = pd.json_normalize(
    respuesta['data'],
    record_path='mediciones',
    meta=['codigo_estacion', 'fecha', ['ubicacion', 'comuna']],
)
api_largo.head(6)

,parametro,valor,unidad,codigo_estacion,fecha,ubicacion.comuna
0,pm25,37.6,ug/m3,EST-001,2026-04-01,Concepción
1,pm10,58.8,ug/m3,EST-001,2026-04-01,Concepción
2,temperatura,13.1,C,EST-001,2026-04-01,Concepción
3,pm25,25.6,ug/m3,EST-001,2026-04-02,Concepción
4,pm10,63.5,ug/m3,EST-001,2026-04-02,Concepción
5,temperatura,15.6,C,EST-001,2026-04-02,Concepción


### ✏️ Actividad 5.1

El resultado anterior está en **formato largo**: una fila por parámetro medido. Para analizar conviene el **formato ancho**: una fila por estación-fecha y una columna por parámetro.

**Escribe el código** que haga esa transformación con `pivot_table`.

In [25]:
# Tu código aquí
tabla_ancha = pd.pivot_table (
    api_largo,
    index=["codigo_estacion", "fecha"],
    columns="parametro",
    values="valor",
    aggfunc="mean"
)
tabla_ancha.head()

parametro                   pm10  pm25  temperatura
codigo_estacion fecha                              
EST-001         2026-04-01  58.8  37.6         13.1
                2026-04-02  63.5  25.6         15.6
                2026-04-03  75.4  29.3         13.3
                2026-04-04  55.9  33.0         11.7
                2026-04-05  79.1  39.1         12.4

## 6 · Fuente 4 — Extracción web

Cuando el dato solo existe publicado en una página, hay que extraerlo. `pandas.read_html` resuelve el caso más común: una tabla HTML.

**Antes de extraer, tres preguntas que no son técnicas:** ¿los términos de uso del sitio lo permiten?, ¿estás generando una carga razonable?, ¿el dato incluye personas? Si alguna respuesta te incomoda, no extraigas.

In [26]:
# Requiere conexión a internet. Si no la tienes, salta esta celda.
URL = 'https://es.wikipedia.org/wiki/Anexo:Comunas_de_Chile'   # [DOCENTE: confirmar o reemplazar]

try:
    tablas = pd.read_html(URL)
    print('Tablas encontradas:', len(tablas))
    web = tablas[0]
    display(web.head())
except Exception as e:
    print('No se pudo extraer:', type(e).__name__, e)
    print('Continúa con el resto del laboratorio.')

No se pudo extraer: HTTPError HTTP Error 403: Forbidden
Continúa con el resto del laboratorio.


### ✏️ Actividad 6.1

Responde en la celda de texto siguiente: **¿qué diferencia hay, para tu proyecto, entre un dato descargado de un portal oficial y uno extraído de una página web?** Piensa en trazabilidad y en qué pasa si mañana la página cambia.

_Tu respuesta aquí:_ La principal diferencia está en la trazabilidad y reproducibilidad de los datos. Un conjunto descargado de un portal oficial suele contar con información sobre su origen, fecha de actualización y metodología, lo que facilita verificar su procedencia. En cambio, los datos extraídos de una página web pueden modificarse o desaparecer sin mantener un registro de versiones, dificultando reproducir el análisis posteriormente. Por ello, es importante documentar la fuente, la fecha de extracción y conservar una copia de los datos utilizados.



## 7 · Volumen: cuando el conjunto no cabe

No todo conjunto se lee entero. Cuando el archivo supera la memoria disponible hay tres salidas: **leer solo las columnas necesarias**, **leer por bloques** o **trabajar con una muestra**.

Una muestra bien construida responde muchas preguntas con una fracción del costo. Una muestra mal construida responde mal todas.

In [27]:
# Leer solo algunas columnas: menos memoria desde el inicio
liviano = pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';',
                      usecols=['estacion', 'fecha', 'pm25'])
print('Memoria completo :', aire.memory_usage(deep=True).sum() / 1024, 'KB')
print('Memoria liviano  :', liviano.memory_usage(deep=True).sum() / 1024, 'KB')

Memoria completo : 335.0009765625 KB
Memoria liviano  : 94.8486328125 KB


In [28]:
# Lectura por bloques: se procesa de a trozos y nunca se carga todo
total, filas = 0, 0
for bloque in pd.read_csv(RUTA + 'S1_Dataset_Guia_CDD.csv', sep=';', chunksize=100):
    valores = pd.to_numeric(bloque['pm25'], errors='coerce').dropna()
    total += valores.sum()
    filas += len(valores)
print(f'Promedio de pm25 calculado por bloques: {total / filas:.2f}')

Promedio de pm25 calculado por bloques: 62.04


In [29]:
# Muestreo aleatorio simple vs. estratificado por estación
simple = aire.sample(n=120, random_state=SEMILLA)
estratificado = (aire.groupby('estacion', group_keys=False)
                     .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))

print('Distribución en la muestra SIMPLE:')
print(simple['estacion'].value_counts().to_string())
print()
print('Distribución en la muestra ESTRATIFICADA:')
print(estratificado['estacion'].value_counts().to_string())

Distribución en la muestra SIMPLE:
estacion
Kingston                  18
Osorno Entre Lagos        18
Coyhanco                  17
Los Ángeles Oriente       17
Coronel Norte             16
Talcahuano                13
Temuco Padre Las Casas    13
Valdivia Centro            8

Distribución en la muestra ESTRATIFICADA:
estacion
Coronel Norte             15
Coyhanco                  15
Kingston                  15
Los Ángeles Oriente       15
Osorno Entre Lagos        15
Talcahuano                15
Temuco Padre Las Casas    15
Valdivia Centro           15


/tmp/ipykernel_1625/748870216.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=15, random_state=SEMILLA)))


### ✏️ Actividad 7.1

Mira las dos distribuciones anteriores. **¿En qué situación la muestra simple te llevaría a una conclusión equivocada?** Escríbelo abajo con un ejemplo concreto de tu propio conjunto de datos.

_Tu respuesta aquí:_ El muestreo aleatorio simple puede llevar a conclusiones equivocadas debido a problemas de representatividad, ya que los registros se seleccionan aleatoriamente y algunas estaciones con menor cantidad de datos podrían quedar insuficientemente representadas. Esto podría generar interpretaciones erróneas sobre los niveles de contaminación. En cambio, el muestreo estratificado permite asegurar la representación de cada estación mediante la selección de registros dentro de cada grupo, favoreciendo un análisis más representativo de las diferencias entre estaciones.



## 8 · Tu conjunto de datos: la elección del proyecto

Abre `S1_Catalogo_Datasets_Sugeridos_CDD.xlsx`, revisa **al menos tres** alternativas y elige una. Registra tu elección en la hoja «Mi elección» del mismo archivo.

Luego descárgalo, cárgalo aquí abajo y ejecuta sobre él el mismo perfilado de la sección 3.

In [30]:

# Semana 1: muestra inicial del proyecto ChileCompra
# Se utiliza un CSV para explorar su estructura y calidad.
# La integración de los 12 archivos se realiza en la semana 2.

mi_dataset = pd.read_csv(
    RUTA + '26940OCLicitacion.csv',
    sep=";",
    encoding="cp1252"
)

print("Dimensiones de la muestra:", mi_dataset.shape)
print("Filas originales:", len(mi_dataset))
print("Órdenes únicas:", mi_dataset["codigoOC"].nunique())

# Revisar las primeras filas
display(mi_dataset.head(5))

# Revisar códigos de orden repetidos
display(
    mi_dataset.loc[
        mi_dataset["codigoOC"].duplicated(keep=False),
        ["codigoOC", "NombreItem", "MontoTotalOC",
         "MontoTotalItem", "CantidadItem"]
    ].sort_values("codigoOC").head(15)
)





Dimensiones de la muestra: (39, 46)
Filas originales: 39
Órdenes únicas: 20


,codigoOC,NroLicitacion,FechaEnvioOC,NombreOC,DescripcionOC,EstadoOC,ProcedenciaOC,CompraCoordinada,MonedaOC,MontoNetoOC,...,DescripcionItem,CantidadItem,UnidadMedida,MonedaItem,MontoNetoItem,DescuentoItem,CargosItem,ImpuestoEspecificoItem,MontoTotalItem,MontoNetoItemCLP
0,624-3-SE26,624-24-LE25,2026-02-03,REPARACIÓN PORTON ELECTRICO GORE,REPARACIÓN PORTON ELECTRICO GORE UBICADO EN ES...,Recepcion Conforme,Licitacion Publica,No,CLP,225200.0,...,"MANTENCIÓN Y/O REPARACION DE OBRAS MENORES, SE...",1.0,Unidad no definida,CLP,225200.0,0.0,0.0,0.0,225200.0,225200.0
1,624-7-SE26,624-16-LE24,2026-02-11,Mantención 190.000 KHFF-29,"En el marco del contrato vigente, se requiere ...",Recepcion Conforme,Licitacion Publica,No,CLP,401168.0,...,MANTENCIÓN CORRECTIVA. VEHICULO PATENTE KHFF-2...,1.0,Unidad no definida,CLP,401168.0,0.0,0.0,0.0,401168.0,401168.0
2,624-10-SE26,624-3-LE26,2026-02-25,ORDEN DE COMPRA DESDE 624-3-LE26,CONTRATACIÓN SOLUCIÓN INTEGRAL PARA GESTIÓN DE...,Recepcion Conforme,Licitacion Publica,No,CLP,36000000.0,...,CONTRATACIÓN SOLUCIÓN INTEGRAL PARA GESTIÓN DE...,1.0,Unidad,CLP,36000000.0,0.0,0.0,0.0,36000000.0,36000000.0
3,624-15-SE26,624-19-LP25,2026-03-09,SERVICIO DE ASEO ADICIONAL,"Se Solicita, la modificación de contrato de pr...",Recepcion Conforme,Licitacion Publica,No,CLP,1446335.0,...,SERVICIO DE PERSONAL DE ASEO DEPENDENCIAS CONS...,6.0,Unidad,CLP,233280.0,0.0,0.0,0.0,1399680.0,1399680.0
4,624-20-SE26,624-13-LE25,2026-03-11,MANTENCIÓN PREVENTIVA EQUIPOS AIRE ACONDICIONADO,SE REQUIERE CONTRATAR LA CANTIDAD DE 5 MANTENC...,Recepcion Conforme,Licitacion Publica,No,CLP,2280000.0,...,5 MANTENCIONES PREVENTIVAS PARA 5 EQUIPOS DE C...,1.0,Unidad,CLP,1210000.0,0.0,0.0,0.0,1210000.0,1210000.0


,codigoOC,NombreItem,MontoTotalOC,MontoTotalItem,CantidadItem
3,624-15-SE26,Servicios de limpieza de edificios,1.721139e+06,1399680.0,6.0
8,624-15-SE26,Servicios de limpieza de edificios,1.721139e+06,46655.0,1.0
4,624-20-SE26,"Servicios de reparación, mantenimiento o repar...",2.713200e+06,1210000.0,1.0
7,624-20-SE26,"Servicios de reparación, mantenimiento o repar...",2.713200e+06,510000.0,1.0
9,624-20-SE26,"Servicios de reparación, mantenimiento o repar...",2.713200e+06,560000.0,1.0
21,624-44-SE26,Carpintería o chapistería de acabados,1.741848e+06,776238.0,1.0
13,624-44-SE26,Carpintería o chapistería de acabados,1.741848e+06,277000.0,1.0
20,624-44-SE26,Carpintería o chapistería de acabados,1.741848e+06,177250.0,1.0
12,624-44-SE26,Carpintería o chapistería de acabados,1.741848e+06,233250.0,1.0
14,624-47-SE26,Aires acondicionados,9.815620e+05,457800.0,1.0


In [31]:
# Perfilado de tu conjunto: repite aquí las cuatro celdas de la sección 3
print("Dimensiones:", mi_dataset.shape)

print(mi_dataset.isna().sum())

print("Duplicados exactos:", mi_dataset.duplicated().sum())

print(mi_dataset[["Institucion", "EstadoOC", "MonedaOC"]].nunique())

Dimensiones: (39, 46)
codigoOC                  0
NroLicitacion             0
FechaEnvioOC              0
NombreOC                  0
DescripcionOC             0
EstadoOC                  0
ProcedenciaOC             0
CompraCoordinada          0
MonedaOC                  0
MontoNetoOC               0
DescuentosOC              0
CargosOC                  0
ImpuestosOC               0
MontoTotalOC              0
ImpuestosOC_CLP           0
MontoNetoOC_CLP           0
MetodoPago                0
TipoDespacho              0
Financiamiento            1
UnidadCompra              0
UnidadCompraRUT           0
RegionUnidadCompra        0
entCode                   0
Institucion               0
Sector                    0
Proveedor                 0
ProveedorRUT              0
ActividadProveedor        0
TamanoProveedor           0
RegionProveedor           0
RubroN1                   0
RubroN2                   0
RubroN3                   0
CodigoProductoONU         0
ONUProducto               

### ✏️ Actividad 8.1 — Pregunta analítica y unidad de análisis

Completa estas dos líneas. Son la base de todo el curso y las vas a necesitar en la Entrega 1.

**Mi pregunta analítica es:**

¿Cómo se distribuyen los montos netos en pesos chilenos de las órdenes de compra de los Gobiernos Regionales de Coquimbo, Metropolitana y Valparaíso durante el primer semestre de 2026, según el mes y el mecanismo de contratación?

**Una fila de mi conjunto representa:**

En los archivos CSV originales, una fila corresponde a un registro asociado a un ítem de una orden de compra, por lo que un mismo código de orden puede aparecer varias veces.

Para el análisis del proyecto, se define como unidad de análisis una orden de compra única, identificada mediante `codigoOC`.

Por lo tanto, durante la semana 2 se integrarán los doce archivos originales y se consolidarán los registros a nivel de orden de compra, evitando duplicar montos debido a la presencia de múltiples ítems.

**Alcance de la exploración inicial:**

En la semana 1 se utiliza un archivo CSV como muestra para examinar la estructura de los datos, sus variables y posibles problemas de calidad. Esta muestra no representa todavía el conjunto completo del proyecto, que se construirá en la etapa posterior de preparación y limpieza.

## 9 · Riesgo ético y legal

Antes de usar tu conjunto, completa la ficha `S1_Ficha_Riesgo_Etico_CDD.docx`. No es un trámite: si tu conjunto contiene datos de personas, lo que decidas ahí condiciona todo lo que puedes hacer en las semanas siguientes.

Esa ficha es el insumo de tu intervención en el **foro de debate** de esta semana y se cierra en la Sección C de la Entrega Final.

## 10 · Repositorio y control de versiones

Tu trabajo tiene que poder reconstruirse. Sube este cuaderno a un repositorio y deja el enlace aquí.

Desde Colab: *Archivo → Guardar una copia en GitHub*. Desde local:

In [32]:
# Ejecuta esto en una terminal, no en el cuaderno:
# git init
# git add S1_Notebook_Base_CDD.ipynb
# git commit -m 'Laboratorio 1: obtención del dato'
# git remote add origin https://github.com/USUARIO/proyecto-ciencia-datos.git
# git push -u origin main


**Enlace a mi repositorio:** _…_

---

## Cierre del laboratorio

Al terminar deberías tener:

- [ ] El entorno funcionando y las versiones declaradas
- [ ] Datos leídos desde las cuatro fuentes
- [ ] El perfilado inicial del conjunto guía, con sus defectos detectados (no corregidos)
- [ ] Tu conjunto de datos elegido y registrado en el catálogo
- [ ] Tu pregunta analítica y tu unidad de análisis escritas
- [ ] La ficha de riesgo ético-legal completada
- [ ] El repositorio creado y este cuaderno versionado

Lo que quede pendiente, llévalo a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 1_